# KCSDB2 무역통계 DB — 주피터 노트북 시작

관세청 무역통계(2007.01–2026.03, 2,753만 거래행) 분석 환경.
이 노트북은 **DB 접속·구조 확인·쿼리 방법과 간단한 기술통계(descriptive)**를 시연한다.
시연은 방법 예시이지 분석 방향이 아니다. 어떤 분석을 할지는 각자 자유롭게 정한다.

출처: 관세청·외교부 공공데이터(공공누리 제1유형 포함). 상세는 저장소 README 참조.

## 0. 사전 준비 (노트북 실행 전)

1. **패키지 설치** (터미널 또는 아래 셀):
   ```
   pip install duckdb pandas matplotlib jupyter
   ```
   재구축이 아니라 분석만 하므로 이 4개면 충분하다. conda·특정 파이썬 버전 불필요.

2. **DB 파일 배치**: 저장소 Releases에서 `kcsdb.duckdb.gz`를 받아 압축을 풀고
   `data/processed/kcsdb.duckdb`에 놓는다. (압축 해제: Windows는 7-Zip 등, Mac/Linux는 `gunzip kcsdb.duckdb.gz`)

3. 이 노트북을 저장소 루트에서 주피터로 연다: `jupyter notebook`

In [ ]:
# (선택) 패키지 설치 — 이미 설치했으면 건너뛴다
# !pip install duckdb pandas matplotlib

## 1. DB 파일 확인 및 접속

DB 경로를 확인한다. 파일이 없으면 위 0단계(Releases에서 받아 배치)를 먼저 한다.

In [ ]:
import os
import duckdb

# 저장소 루트에서 실행 가정. 경로가 다르면 수정한다.
DB_PATH = os.path.join("data", "processed", "kcsdb.duckdb")

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"DB 없음: {DB_PATH}\n"
        "Releases에서 kcsdb.duckdb.gz를 받아 압축 해제 후 data/processed/ 에 놓으세요."
    )

print("DB 크기:", os.path.getsize(DB_PATH)//1024//1024, "MB")
con = duckdb.connect(DB_PATH, read_only=True)
print("접속 완료")

## 2. 테이블·기간 확인

In [ ]:
print(con.sql("SHOW TABLES").df())
print("\nfact_trade 행수:", con.sql("SELECT COUNT(*) FROM fact_trade").fetchone()[0])
print("기간:", con.sql("SELECT MIN(yyyymm), MAX(yyyymm) FROM fact_trade").fetchone())

## 3. 스키마 확인

In [ ]:
for t in ["fact_trade", "fact_total", "dim_country", "dim_hs10", "dim_hs6_concordance"]:
    print(f"=== {t} ===")
    print(con.sql(f"DESCRIBE {t}").df().to_string(index=False))
    print()

## 4. 기술통계 시연 (descriptive)

아래는 **방법 시연**이다. 특정 가설·모형을 함축하지 않고 "데이터가 무엇을 담았는가"만 보인다.
**단위: 금액 미화 천 달러, 중량 kg.**

주의(반드시 읽을 것):
- HS 시계열을 코드 동일성으로 이으면 2022 개정 경계에서 왜곡된다(dim_hs6_concordance 필요).
- 202603이 종점. 마지막 해/달을 추세로 읽지 말 것.
- 음수 중량 19행은 관세청 정정 원본. 단가 계산 시 이상치.

In [ ]:
import matplotlib.pyplot as plt

# (1) 연도별 총 수출·수입 (천 달러) — 2026은 3월까지만이므로 제외
df1 = con.sql("""
    SELECT yyyymm // 100 AS year,
           SUM(exp_dlr) AS exports,
           SUM(imp_dlr) AS imports
    FROM fact_trade
    WHERE yyyymm // 100 < 2026
    GROUP BY 1 ORDER BY 1
""").df()
print(df1)

plt.figure(figsize=(9,4))
plt.plot(df1['year'], df1['exports']/1e6, marker='o', label='Exports')
plt.plot(df1['year'], df1['imports']/1e6, marker='s', label='Imports')
plt.ylabel('Billion USD'); plt.xlabel('Year')
plt.title('Korea Annual Trade (2007-2025, excl. partial 2026)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# (2) 상위 교역국 (전 기간 누적 교역액) — dim_country 조인
df2 = con.sql("""
    SELECT d.name_ko_kcs AS country,
           SUM(f.exp_dlr + f.imp_dlr) AS trade
    FROM fact_trade f
    LEFT JOIN dim_country d ON f.stat_cd = d.stat_cd
    GROUP BY 1 ORDER BY trade DESC LIMIT 10
""").df()
print(df2)

plt.figure(figsize=(9,4))
plt.barh(df2['country'][::-1], df2['trade'][::-1]/1e6)
plt.xlabel('Billion USD (cumulative 2007-2026.03)')
plt.title('Top 10 Trade Partners')
plt.tight_layout(); plt.show()

In [ ]:
# (3) HS2 대분류별 수출 상위 10 — hs10을 hs2로 절단(dim 없이 SUBSTR)
df3 = con.sql("""
    SELECT SUBSTR(hs10,1,2) AS hs2,
           SUM(exp_dlr) AS exports
    FROM fact_trade
    GROUP BY 1 ORDER BY exports DESC LIMIT 10
""").df()
print(df3)

plt.figure(figsize=(9,4))
plt.bar(df3['hs2'], df3['exports']/1e6)
plt.ylabel('Billion USD'); plt.xlabel('HS2 code')
plt.title('Top 10 HS2 Categories by Export (cumulative)')
plt.tight_layout(); plt.show()
print("\nHS2 코드의 품목명은 dim_hs10이 hs10 단위라 직접 없음.")
print("필요시 HS 분류표 참조(예: 85=전기기기, 87=차량). 분석 목적에 따라 해석.")

In [ ]:
# (4) 연도별 무역수지 (수출-수입)
df4 = con.sql("""
    SELECT yyyymm // 100 AS year,
           SUM(exp_dlr - imp_dlr) AS balance
    FROM fact_trade
    WHERE yyyymm // 100 < 2026
    GROUP BY 1 ORDER BY 1
""").df()
print(df4)

plt.figure(figsize=(9,4))
colors = ['crimson' if b < 0 else 'steelblue' for b in df4['balance']]
plt.bar(df4['year'], df4['balance']/1e6, color=colors)
plt.axhline(0, color='black', lw=0.8)
plt.ylabel('Billion USD'); plt.xlabel('Year')
plt.title('Korea Annual Trade Balance (2007-2025)')
plt.tight_layout(); plt.show()

## 5. HS 개정 연결 사용법 (concordance 조인 시연)

In [ ]:
# 과거 hs6를 hs2022 기준으로 환산. 다대다이므로 실제 분석 시 중복 처리 필요.
df5 = con.sql("""
    SELECT past_version,
           COUNT(DISTINCT hs_past) AS past_codes,
           COUNT(DISTINCT hs2022) AS mapped_2022
    FROM dim_hs6_concordance
    WHERE relation = 'mapped'
    GROUP BY 1 ORDER BY 1
""").df()
print(df5)

## 6. 분석 시 주의 (필독)

- **대용량 결과를 .df()로 통째 가져오지 말 것.** 집계를 SQL 안에서 끝내고 소형 결과만.
- **HS 개정 연결**: 코드 동일성만으로 시계열을 이으면 2022 개정 품목이 끊긴다. dim_hs6_concordance로 통일.
- **음수 중량 19행**: 관세청 정정 원본. 단가 계산 시 이상치.
- **범위 경계**: 202603까지. 2026은 부분년(1-3월)이므로 연도 비교 시 제외했다.
- 상세 함정은 저장소 docs/세션_발견_노트.md 참조.

## 7. 마무리

분석이 끝나면 연결을 닫는다.

In [ ]:
con.close()